# ePADDY Confusion Matrix — Normalized (%) with Custom Palette

Loads a trained YOLO classification model, runs validation, extracts the confusion matrix,
normalizes it to percentages, and plots it with a custom hex-based colormap.

**Edit the CONFIG cell** with your model path, data path, variety, and season.

In [ ]:
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

## CONFIG — edit these

In [ ]:
# --- Paths ---
MODEL_PATH = "/path/to/folder" # trained classification model weights
DATA_PATH  = "/path/to/folder" # validation dataset root (YOLO classification folder structure)

# --- Plot identity ---
DATASET_TAG = "WS2025"                  # e.g. DS2025 or WS2025
VARIETY     = "IR24"                    # e.g. IR24 or SL8H

# --- Custom palettes (hex) ---
# Dry Season
DS_PALETTE = {
    "IR24": ["#FFFFFF", "#1f77b4"],   # white -> blue
    "SL8H":  ["#FFFFFF", "#ff7f0e"],   # white -> orange
}

# Wet Season
WS_PALETTE = {
    "IR24": ["#FFFFFF", "#2ca02c"],   # white -> green
    "SL8H":  ["#FFFFFF", "#9467bd"],   # white -> purple
}

# Pick palette set based on dataset tag (DS = Dry Season, WS = Wet Season)
PALETTE_SET = DS_PALETTE if DATASET_TAG.startswith("DS") else WS_PALETTE
# PALETTE_SET = INSECT_PALETTE
HEX_COLORS = PALETTE_SET[VARIETY]

print(f"Using palette for {VARIETY} ({DATASET_TAG}): {HEX_COLORS}")

## Rename the Dataset

## Run YOLO validation to get the confusion matrix

In [ ]:
import ultralytics.utils.plotting as plotting

model = YOLO(MODEL_PATH)
print(DATA_PATH)

# --- Recolor val batch labels by group (Control / Medium / High) ---
GROUP_COLORS = {
    "Control": (57, 255, 20),    # neon green
    "Medium":  (255, 255, 0),    # bright yellow
    "High":    (255, 30, 30),    # bright red
}
DEFAULT = (255, 255, 255)        # fallback for unmatched names

names = model.names              # {0: 'Control_08DAI', ...}
n = max(names) + 1
new_palette = [DEFAULT] * n
for idx, cls_name in names.items():
    for group, rgb in GROUP_COLORS.items():
        if cls_name.startswith(group):
            new_palette[idx] = rgb
            break
plotting.colors.palette = new_palette
plotting.colors.n = len(new_palette)
# -------------------------------------------------------------------

metrics = model.val(data=DATA_PATH, split="val")

# Raw confusion matrix (counts), shape: (num_classes, num_classes)
cm_raw = metrics.confusion_matrix.matrix
class_names = list(model.names.values())

print("Classes:", class_names)
print("Raw confusion matrix shape:", cm_raw.shape)
print(cm_raw)

## Normalize to percentages (per true-class column)

In [ ]:
# Normalize each column (true class) so values sum to 1 (i.e. 100%)
col_sums = cm_raw.sum(axis=0, keepdims=True)
col_sums[col_sums == 0] = 1  # avoid divide-by-zero for empty classes
cm_norm = (cm_raw / col_sums)*100

print("Normalized confusion matrix (fractions, columns sum to 1):")
print(np.round(cm_norm, 2))

## Build custom colormap and plot

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list("custom_cmap", HEX_COLORS)

fig, ax = plt.subplots(figsize=(12, 10))

# Color only — no seaborn annotation. Zero cells render white automatically
# since the colormap starts at white.
sns.heatmap(
    cm_norm,
    annot=False,
    cmap=custom_cmap,
    cbar=True,
    square=True,
    linewidths=0.5,
    linecolor="white",
    xticklabels=class_names,
    yticklabels=class_names,
    vmin=0,
    vmax=100,
    ax=ax,
)

# Manually add % labels so every nonzero cell is reliably labeled
n_rows, n_cols = cm_norm.shape
for i in range(n_rows):
    for j in range(n_cols):
        val = cm_norm[i, j]
        if val > 0:
            text_color = "Black"  # contrast vs cell
            ax.text(j + 0.5, i + 0.5, f"{val:.2f}",
                    ha="center", va="center", color=text_color, fontsize=10)

ax.set_title(f"{DATASET_TAG} | {VARIETY} | Confusion Matrix (%)", fontsize=14, fontfamily="DejaVu Sans")
ax.set_xlabel("True", fontsize=14, fontfamily="DejaVu Sans")
ax.set_ylabel("Predicted", fontsize=14, fontfamily="DejaVu Sans")

plt.xticks(rotation=90, fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()

out_path = f"{DATASET_TAG}_{VARIETY}_confusion_matrix_renamed.png"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")

## Classification metrics (Accuracy, Precision, Recall, F1, Support) → .txt

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# YOLO confusion matrix orientation is matrix[predicted, true].
# Append a 'background' label if the matrix carries the extra row/col.
labels = list(class_names)
if cm_raw.shape[0] == len(class_names) + 1:
    labels = list(class_names) + ["background"]

# Reconstruct y_true / y_pred from the raw counts
y_true, y_pred = [], []
for pred in range(cm_raw.shape[0]):
    for true in range(cm_raw.shape[1]):
        c = int(cm_raw[pred, true])
        if c:
            y_true += [true] * c
            y_pred += [pred] * c
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Report only classes that actually appear (drops empty background row/col)
present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
present_names = [labels[i] for i in present]

report = classification_report(
    y_true, y_pred,
    labels=present,
    target_names=present_names,
    digits=4,
    zero_division=0,
)
acc = accuracy_score(y_true, y_pred)

# Compose the text output
header = f"{DATASET_TAG} | {VARIETY} | Classification Metrics\n"
header += "=" * len(header.strip()) + "\n\n"
body = report + f"\nOverall Accuracy: {acc:.4f}\n"
full_text = header + body

# Save to .txt
txt_path = f"{DATASET_TAG}_{VARIETY}_classification_metrics.txt"
with open(txt_path, "w") as f:
    f.write(full_text)

print(full_text)
print(f"Saved: {txt_path}")

In [ ]:
import os
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SPLITS = ["train", "val", "test"]

def count_images(folder):
    p = Path(folder)
    if not p.is_dir():
        return 0
    return sum(1 for f in p.iterdir()
               if f.is_file() and f.suffix.lower() in IMG_EXTS)

# Discover class folders present across splits
disk_classes = set()
for sp in SPLITS:
    sp_dir = Path(DATA_PATH) / sp
    if sp_dir.is_dir():
        disk_classes |= {d.name for d in sp_dir.iterdir() if d.is_dir()}

ordered = [c for c in class_names if c in disk_classes] + \
          sorted(disk_classes - set(class_names))

counts = {sp: {c: count_images(Path(DATA_PATH) / sp / c) for c in ordered}
          for sp in SPLITS}

# Line format: "train - Control_T1 = 25 images"
lines = [f"{DATASET_TAG} | {VARIETY} | Dataset image counts", ""]
for sp in SPLITS:
    for c in ordered:
        lines.append(f"{sp} - {c} = {counts[sp][c]} images")
    lines.append("")

table_text = "\n".join(lines)
print(table_text)

counts_path = f"{DATASET_TAG}_{VARIETY}_dataset_counts.txt"
with open(counts_path, "w") as f:
    f.write(table_text + "\n")
print(f"Saved: {counts_path}")